<small>📌 <b>먼저 내 노트로 저장하기</b> — ① 맨 위 <b>파일 → Drive에 사본 저장</b> (안 하면 쓴 게 사라져요) ② 사본 오른쪽 위 <b>공유</b> → 일반 액세스 「링크가 있는 모든 사용자」 → <b>링크 복사</b> → 수행평가 화면 「📓 내 Colab 실습 노트」 칸에 붙여넣고 보관.</small><br><img src="https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/colab_save.png" width="300"> <img src="https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/colab_share.png" width="300">

# 📈 내가 정한 주제 — 데이터 분석 빈칸 실습 (내 CSV 로)

카드에 없는 **내 주제**를 **내가 찾은 데이터**로 분석해요. 순서는 카드 노트와 같아요:
1. 올리기 → 2. 열 고르기 → 3. 결측치 → 4. 이상치 → 5. 요약 → 6. 그래프 → 7. pairplot·상관·heatmap → 8. 나누기 → 8-1. 정규화 → 9. 회귀 모델 → 10. 내 생각

`______` 빈칸에 알맞은 **함수 이름이나 숫자**를 넣고 ▶ 실행해요. 막히면 **오픈북** — 클래스룸에 올린 그간의 실습 파일을 펴 보고 같은 코드를 찾아 써요.

> 다 하면 「파일 → 드라이브에 사본 저장」 → 공유 링크를 수행평가 화면 「📓 내 Colab 실습 노트」 칸에 보관해요.

In [ ]:
# 0. 준비 — 그대로 실행해요 (한글 그래프 글꼴)
!pip -q install koreanize-matplotlib
import matplotlib.pyplot as plt
import koreanize_matplotlib

## 1. 내 데이터 올리기
내가 찾은 CSV 파일을 올려요. ▶ 실행하면 「파일 선택」 단추가 나와요.

- 엑셀 파일이면 먼저 「다른 이름으로 저장 → CSV UTF-8」로 바꿔요.
- 구글 스프레드시트면 「파일 → 다운로드 → CSV」로 받아 올려요.
- 공공데이터포털(data.go.kr)·KOSIS 에서 받은 CSV 가 한글이 깨지면 아래 `encoding="cp949"` 줄을 써요.

In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()                    # 파일 선택 창이 떠요
name = next(iter(up))
try:
    df = pd.______(name)               # 힌트: CSV 를 읽는 함수
except UnicodeDecodeError:
    df = pd.read_csv(name, encoding="cp949")      # 한글 깨짐(공공데이터) 대비
print(df.shape)
df.______()                            # 힌트: 앞 5줄만 보기

## 2. 열 살펴보고 내 주제에 맞게 고르기
아래 목록에서 **결과(예측하고 싶은 것)** 열 하나와 **원인일 것 같은 숫자 열** 몇 개를 골라 이름을 그대로 적어요.

In [ ]:
df.info()      # 열 이름·빈칸 아닌 개수·자료형(숫자는 int/float)

In [ ]:
Y = "결과 열 이름"                    # ← 바꿔요: 예측하고 싶은 열
X_COLS = ["원인 열1", "원인 열2"]      # ← 바꿔요: 숫자 열 몇 개
for c in [Y] + X_COLS:
    assert c in df.columns, f"'{c}' 라는 열이 없어요 — 위 목록의 이름을 그대로 복사해요"
df = df[[Y] + X_COLS]
for c in df.columns:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce')      # '1,234' 같은 글자 숫자를 숫자로
df.head()

## 3. 결측치(빈칸) 처리
📖 **오픈북** — 수업 결측치 실습 `isnull().sum()`, `dropna()`

In [ ]:
print(df.______().sum())      # 힌트: 빈칸인지 True/False — 열마다 개수
df = df.______()              # 힌트: 빈칸 있는 줄 지우기
print('빈칸 지운 뒤', df.shape)

## 4. 이상치 거르기 (IQR)
Q1(25%)·Q3(75%) 사이 폭(IQR)의 1.5배를 넘게 벗어난 값은 이상치로 보고 걸러요. 값 종류가 적은 열(0/1 등)은 건너뛰어요.

In [ ]:
for col in df.columns:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(______)      # 힌트: 75% 는 소수로?
    iqr = q3 - q1
    if iqr == 0 or df[col].nunique() <= 5: continue
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]
print('거른 뒤', df.shape)

## 5. 요약 통계

In [ ]:
df.______()      # 힌트: 평균·표준편차·최소·최대를 한 번에

## 6. 그래프 — 원인 열과 결과의 관계

In [ ]:
for c in X_COLS:
    plt.______(df[c], df[Y], alpha=0.5)      # 힌트: 점 그래프(산점도)
    plt.xlabel(c); plt.ylabel(Y); plt.title(f'{c} 와 {Y}'); plt.show()

## 7. 상관 — 결과와 가장 관계 깊은 열은?
1 에 가까우면 같이 커지고, -1 에 가까우면 반대로 움직여요. 0 근처면 거의 관계 없음.

### 7-1. pairplot — 열끼리 한꺼번에 점 그래프
모든 짝의 점 그래프를 한 번에 봐요. 결과 줄에서 점들이 비스듬한 선을 이루는 열이 관계가 큰 열이에요. 📖 **오픈북** — 「#16 펭귄-분류」 `sns.pairplot(df)`

In [ ]:
import seaborn as sns
sns.______(df.sample(min(len(df), 300), random_state=0), corner=True, plot_kws={'s': 12})      # 힌트: 짝마다 점 그래프 (p…plot)
plt.show()

### 7-2. 결과와의 상관계수

In [ ]:
print(df.______()[Y].sort_values())      # 힌트: 상관계수 표

### 7-3. heatmap — 상관계수 표를 색으로
빨갈수록 같이 커지고, 파랄수록 반대로 움직여요. 📖 **오픈북** — 「#14 Iris 데이터 전처리」 `sns.heatmap(df.corr(), annot=True)`

In [ ]:
plt.figure(figsize=(7, 5))
sns.______(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)      # 힌트: 색 표 (h…map)
plt.show()

## 8. 학습용·시험용 나누기

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df[X_COLS], df[Y]
X_train, X_test, y_train, y_test = ______(X, y, test_size=0.3, random_state=42)      # 힌트: 나누는 함수

## 8-1. 정규화 — 숫자 크기를 0~1 로
가장 작은 값은 0, 가장 큰 값은 1 이 되게 바꿔요. 기준은 학습용에서만 구하고 시험용도 같은 기준으로. 📖 **오픈북** — 「MNIST 딥러닝」 정규화

In [ ]:
mn, mx = X_train.min(), X_train.______()      # 힌트: 가장 큰 값 (가장 작은 값은 min)
rng = (mx - mn).replace(0, 1)
X_train, X_test = (X_train - mn) / rng, (X_test - mn) / rng
X_train.describe().loc[['min', 'max']]

## 9. 선형 회귀 모델

In [ ]:
from sklearn.linear_model import LinearRegression
model = ______()                       # 힌트: 선형 회귀
model.______(X_train, y_train)          # 힌트: 학습시키기
print('시험 점수(R²):', round(model.score(X_test, y_test), 3))
print(dict(zip(X_COLS, model.coef_.round(3))))      # 원인 열이 1 늘 때 결과가 얼마나 바뀌나

## 10. 내 생각 (보고서 C·D 칸에 옮겨 써요)
- 결과와 가장 관계 깊은 열은 ___ 이었다. 상관계수는 ___.
- 모델 점수(R²)는 ___ 이라서 ___ (잘 맞았다 / 덜 맞았다) — 이유는 ___ 일 것 같다.
- 상관이 있다고 원인이라고 할 수는 없다. 다른 원인 ___ 도 생각해 봐야 한다.